#### **03-rl**

A vectorized environment over the same engine. Hundreds of independent
environments walk decorrelated stretches of one shared candle series, stepped
together with the GIL released.

**You control three things and the engine controls none of them.** What the
agent sees, what it is paid, and what its actions mean. The engine never
computes a reward, which is what keeps the same fill model usable outside
reinforcement learning at all.

The policy that steps it first is random, because that is the line a trained
agent has to clear. The last two sections train one with PPO and put the two
side by side on bars it never saw.

#### **Setup**

The environment is built before this cell runs. `docker compose up` installs
both libraries and starts a router.

Gymnasium arrives with emsl rather than as an extra, because the vector
env subclasses it. `stable-baselines3` comes from the `sb3` extra, which this
image installs because the last two sections train a policy. Torch is
installed separately, as the CPU build from pytorch's own index: the default
wheel carries a CUDA runtime of several gigabytes that nothing here can
reach.

In [1]:
import os
import urllib.request

ROUTER_URL = os.environ.get(key="ROUTER_URL", default="http://127.0.0.1:8040")

try:
    import numpy
    import gymnasium
    import torch
    import stable_baselines3
    import emsl
    import emsl.sb3
    import exchange_router_client
except ImportError as error:
    raise ImportError(f"{error.name} is missing; run 'docker compose up' from "
                      f"this repository, or pip install -r requirements.txt") from None


def router_is_up(url):
    try:
        with urllib.request.urlopen(url=f"{url}/status", timeout=2.0) as response:
            return response.status == 200
    except Exception:
        return False


if not router_is_up(url=ROUTER_URL):
    raise RuntimeError(f"no router answering on {ROUTER_URL}; run 'docker compose "
                       f"up' from this repository, or start one yourself and point "
                       f"ROUTER_URL at it")

client = exchange_router_client.ExchangeRouterClient(base_url=ROUTER_URL)

print("service    ", client.get_version(), "at", ROUTER_URL)
print("emsl       ", emsl.__version__)
print("gymnasium  ", gymnasium.__version__)
print("sb3        ", stable_baselines3.__version__)
print("torch      ", torch.__version__)

service     2.5.6 at http://exchange-router-service:8040
emsl        1.3.1
gymnasium   1.3.0
sb3         2.9.0
torch       2.14.0+cpu


#### **The candles**

One year of hourly bars, pinned so the run is over the same bars every time.
**`start` is the end of the window and the router walks backwards from it**, so
this asks for the year that ends on 1 January 2026, which is 2025.

These are linear perpetual bars rather than spot, which is what makes funding
and liquidation possible further down.

A frame carries metadata, not just columns. `candles.attrs` holds what is
constant across the request: the venue, the symbol, the quote asset, the unit
volume is counted in, and the schema number. Read it before anything is
handed to an agent, because **the observation built in the next section is
your own array and carries none of it.**

In [2]:
import datetime

EXCHANGE = "binance"
MARKET   = "linear"
SYMBOL   = "BTCUSDT"
INTERVAL = "1h"

ANCHOR = int(datetime.datetime(year=2026, month=1, day=1,
                               tzinfo=datetime.timezone.utc).timestamp() * 1000)
BARS   = 8760

candles = client.get_candles(
    exchange=EXCHANGE,
    market_type=MARKET,
    symbol=SYMBOL,
    interval=INTERVAL,
    limit=BARS,
    start=ANCHOR,
)

print(f"{len(candles)} bars, {candles.index[0]} to {candles.index[-1]}")
print(f"columns  {', '.join(candles.columns)}")

for key in ("exchange", "market_type", "symbol", "quote", "volume_unit",
            "schema_version"):
    print(f"  {key:<16} {candles.attrs[key]}")

8760 bars, 2025-01-01 01:00:00+00:00 to 2026-01-01 00:00:00+00:00
columns  open, high, low, close, volume, volume_usd
  exchange         binance
  market_type      linear
  symbol           BTCUSDT
  quote            USDT
  volume_unit      base
  schema_version   3


#### **What the agent sees**

The observation is a window of rows, shape `(num_envs, window, F)`, float32.
Pass `features` of shape `(T, F)` and the agent sees a window of your
indicators; leave it out and it sees the raw `(window, 5)` candles.

Three features here, chosen so none of them is a price. **An agent handed raw
prices learns the price**, and a level that never recurs is a level it can
memorise instead of a pattern it can use. A bounded oscillator, a standardised
distance from a mean, and a volatility reading in percent all survive the asset
moving to a range it has never traded in.

**The environment has no warm-up concept, and this is the trap.** Every
indicator has leading NaN, environments start at random offsets, and the
earliest legal offset is only `window - 1`. So an environment can open onto a
window that is still part NaN, and nothing raises. Trim both the bars and the
features by the longest lookback before building anything.

In [3]:
WARM = 96

closes = candles["close"].to_numpy()
highs  = candles["high"].to_numpy()
lows   = candles["low"].to_numpy()

features = numpy.column_stack([
    emsl.ta.rsi(values=closes, length=14),
    emsl.ta.zscore(values=closes, length=WARM),
    emsl.ta.natr(high=highs, low=lows, close=closes, length=14),
])

print(f"before trimming: {int(numpy.isnan(features).any(axis=1).sum())} "
      f"rows carry a NaN")

bars     = candles.iloc[WARM:]
features = features[WARM:]

print(f"after trimming:  {int(numpy.isnan(features).sum())} NaN left")
print(f"bars {bars.shape}, features {features.shape}")

before trimming: 95 rows carry a NaN
after trimming:  0 NaN left
bars (8664, 6), features (8664, 3)


#### **The environment**

`reward_fn(state, prev)` runs once per step over arrays, not once per
environment. Both arguments expose the account as `(num_envs,)` arrays:
`equity`, `position`, `unrealized_pnl`, `realized_pnl`, `mark_price` and
`funding_paid`. Returning one value per environment is what keeps a reward off
the per-environment Python path. Leave it out and the reward is the change in
equity.

The reward below pays for equity gained and charges for size held, which is the
cheapest way to stop an agent learning that maximum leverage is free.

**A cost knob given a `(low, high)` pair is drawn per environment** rather than
shared, so the batch trains across a spread of cost regimes instead of one lucky
setting. The draw comes from the constructor seed and is **fixed for that
environment's life**, surviving autoresets: it is heterogeneity across workers,
not noise across episodes.

This is a perpetual venue, so funding is charged and an account can be
liquidated, which makes `terminations` a real outcome rather than permanently
false. At this trade size it stays rare.

In [4]:
VENUE = emsl.Market(
    kind="perp",
    quote=10_000.0,
    fee_taker=(0.0002, 0.0008),    #  each env draws its own, once, for its life
    slippage_bps=(0.0, 4.0),
    leverage=3.0,
    funding_rate=0.0001,
    funding_interval=8,
)

CARRY = 0.01


def reward_fn(state, prev):
    return (state.equity - prev.equity) - CARRY * numpy.abs(state.position)


env = VENUE.env(
    data=bars,
    features=features,
    num_envs=256,
    window=32,
    reward_fn=reward_fn,
    trade_size=0.05,
    episode_len=512,
    seed=0,
)

observation, info = env.reset(seed=0)

print(f"observation      {observation.shape} {observation.dtype}")
print(f"one env sees     {env.single_observation_space.shape}")
print(f"one env emits    {env.single_action_space}")

observation      (256, 32, 3) float32
one env sees     (32, 3)
one env emits    Discrete(3)


#### **Stepping it**

The default action space is `Discrete(3)`: hold, buy `trade_size`, sell
`trade_size`. Pass `action_fn(actions, state)` with an `action_space` to decode
your own, for instance a continuous target position where the size traded is the
difference between the target and what is held.

**An episode ends two different ways and they are not the same event.**
`terminations` is true when equity reached zero, which on a perpetual is a
liquidation. `truncations` is true when the environment ran out of bars or hit
`episode_len`. One is a failure, the other is a clock.

**Finished environments reset on the same step**, so the observation returned
for a done environment is already the first of its next episode. The real final
observation and equity are handed back in `infos` under `final_observation` and
`final_equity`, with a boolean mask each.

A discounted return computed without masking on `terminations | truncations`
**leaks across the episode boundary**, quietly, and produces a number that
looks fine.

In [5]:
STEPS = 400

policy = numpy.random.default_rng(seed=0)

rewards     = []
liquidated  = 0
ran_out     = 0
final_seen  = []

for _ in range(STEPS):
    actions = policy.integers(low=0, high=3, size=env.num_envs)

    observation, reward, terminations, truncations, infos = env.step(actions)

    rewards.append(reward)
    liquidated += int(terminations.sum())
    ran_out    += int(truncations.sum())

    if "final_equity" in infos:
        ended = infos["final_equity"][infos["_final_equity"]]
        final_seen.extend(ended.tolist())

rewards = numpy.stack(rewards)

print(f"{STEPS} steps of {env.num_envs} environments "
      f"= {STEPS * env.num_envs:,} bars simulated")
print(f"reward         {rewards.shape} {rewards.dtype}, mean {rewards.mean():,.4f}")
print(f"terminated     {liquidated}   (equity reached zero)")
print(f"truncated      {ran_out}   (out of bars, or episode_len)")

if final_seen:
    ended = numpy.array(final_seen)
    print(f"\nfinal equity over {len(ended)} finished episodes")
    print(f"  median       {numpy.median(ended):>12,.2f}")
    print(f"  worst        {ended.min():>12,.2f}")
    print(f"  best         {ended.max():>12,.2f}")
    print(f"  started at   {10_000.0:>12,.2f}")

400 steps of 256 environments = 102,400 bars simulated
reward         (400, 256) float32, mean -2.0568
terminated     0   (equity reached zero)
truncated      13   (out of bars, or episode_len)

final equity over 13 finished episodes
  median           9,391.73
  worst            8,875.08
  best            10,499.74
  started at      10,000.00


#### **Training a policy**

Stable-Baselines3 does not take a Gymnasium vector env. It wants its own
`VecEnv`, so emsl ships `EmslVecEnv`, which is a remap rather than a second
vectorization: the batch keeps the observation, the reward and the action
decoding, and the same-step autoreset above is already the contract SB3 expects,
so a finished environment's real final observation becomes its
`terminal_observation`.

The training environment is a fresh one over the first seventy percent. The
one built above walks the whole series, which is right for showing how it
steps and wrong for training something you then evaluate. An agent has far
more freedom to fit bars than a two-length crossover does, so a holdout
matters more here.

`n_steps=64` across 128 environments is 8,192 transitions per update, and
`total_timesteps` counts the whole batch, so half a million of them is under two
thousand steps per environment. The cell takes about a minute on a laptop CPU.

SB3 is not required for any of this. The environment is already a Gymnasium
vector env, so a single-file policy gradient reads the batched observation and
writes a batched action. The only emsl-specific line is the one that stops a
discounted return crossing an episode boundary:

```python
masks.append(torch.as_tensor(~(terminations | truncations), dtype=torch.float32))
```

In [6]:
SPLIT = int(len(bars) * 0.7)

train_env = VENUE.env(
    data=bars.iloc[:SPLIT],
    features=features[:SPLIT],
    num_envs=128,
    window=32,
    reward_fn=reward_fn,
    trade_size=0.05,
    episode_len=512,
    seed=0,
)

model = stable_baselines3.PPO(
    policy="MlpPolicy",
    env=emsl.sb3.EmslVecEnv(env=train_env),
    n_steps=64,
    batch_size=1024,
    n_epochs=4,
    seed=0,
    verbose=0,
)

model.learn(total_timesteps=500_000)

print(f"trained on {SPLIT} bars, {len(bars) - SPLIT} held back")
print(f"sees {train_env.single_observation_space.shape}, "
      f"emits {train_env.single_action_space}")

trained on 6064 bars, 2600 held back
sees (32, 3), emits Discrete(3)


#### **Against the baseline**

Training keeps no trade log, because the batch is built without one, so what
a rollout hands back is the equity each episode ended on. Both policies are
stepped through the same construction over the held-back bars, which is what
makes the two comparable.

They are not bar for bar identical. Environments open at random offsets and
draw fresh ones whenever an episode ends, and when an episode ends depends on
what the policy did. So this compares two distributions over many episodes
rather than two runs of one episode.

`deterministic=True` takes the policy's best action rather than sampling from
it, which is what you want when measuring rather than exploring.

In [7]:
def rollout(policy, steps=600, seed=1):
    env = VENUE.env(
        data=bars.iloc[SPLIT:],
        features=features[SPLIT:],
        num_envs=128,
        window=32,
        reward_fn=reward_fn,
        trade_size=0.05,
        episode_len=512,
        seed=seed,
    )

    observation, _info = env.reset(seed=seed)
    finished = []
    mix = numpy.zeros(shape=3, dtype=numpy.int64)

    for _ in range(steps):
        actions = policy(observation=observation)

        for choice in range(3):
            mix[choice] += int((actions == choice).sum())

        observation, _reward, _terms, _truncs, infos = env.step(actions)

        if "final_equity" in infos:
            finished.extend(infos["final_equity"][infos["_final_equity"]].tolist())

    env.close()
    return numpy.array(finished), mix / mix.sum()


chance = numpy.random.default_rng(seed=0)

POLICIES = (
    ("random", lambda observation: chance.integers(low=0, high=3,
                                                   size=len(observation))),
    ("trained", lambda observation: model.predict(observation=observation,
                                                  deterministic=True)[0]),
)

for name, policy in POLICIES:
    finished, mix = rollout(policy=policy)

    print(f"{name:<8} {len(finished):>4} episodes   "
          f"median {numpy.median(finished):>10,.2f}   "
          f"worst {finished.min():>10,.2f}   best {finished.max():>10,.2f}")
    print(f"{'':<8} hold {mix[0]:.3f}   buy {mix[1]:.3f}   sell {mix[2]:.3f}\n")

random    140 episodes   median   9,075.99   worst   4,878.43   best  14,417.67
         hold 0.332   buy 0.335   sell 0.333



trained   140 episodes   median  10,000.00   worst  10,000.00   best  10,000.00
         hold 1.000   buy 0.000   sell 0.000



#### **Closing**

A random policy is a baseline. It pays the spread on every trade it
makes, so its median outcome is roughly the starting balance minus the cost
of churning. Any agent that does not clear that line has learned nothing,
which is a number to have before a training run rather than after one.

**Clearing it is not the same as finding an edge.** The agent above beats the
baseline by holding on every step of the holdout, which is the correct answer
to the question it was asked: with a fee drawn per environment, slippage on
every fill and a charge on size held, not trading beats trading at random. It
found the cost structure rather than a signal, and three features against half
a million steps is a demonstration of the loop rather than a serious attempt
at one.

The three things you control are the three places the work is. The
observation decides what is learnable, the reward decides what is worth
learning, and the action decoding decides what can be expressed. The fill model
underneath is the same one a backtest uses, which is what makes an agent's
result comparable to a rule's.

**Mask on `terminations | truncations`.** The same-step autoreset is the
convention most single-file RL code was written against, and it is the one thing
here that fails silently rather than loudly.

`client.close()` shuts down the background thread the client runs its async loop
on, and its connection pool.

In [8]:
env.close()
client.close()